# InSAR Monitoring of Arctic Landfast Sea-Ice Deformation — Cambridge Bay, Nunavut
## Winter 2018–2019
### Sentinel-1-only replication of Chen, Z. *et al.* (2021), *Remote Sensing* **13**, 4570

Run top to bottom. The companion notebook `InSAR_CambridgeBay_FINAL_2021.ipynb` does
winter 2020–2021; run both, then **S13/S14** compare them.

**Method summary**

| Step | What it does |
|---|---|
| S3 | Water mask, then P1–P4 placed by **ice surface class** — P1/P3 smooth ice, P2 rough ice, P4 the smooth/rough transition (paper §4.2), with distance-from-shore as the depth proxy |
| S3b | **Deramping** — fits a plane to each interferogram over stable coherent land (which cannot deform) and subtracts it. Compensates for the split-spectrum ionospheric correction that HyP3 does not do |
| S9/S10 | SBAS by displacement increments, solved with `pinv` (SVD minimum-norm), per point and per pixel |
| S10 | 2D decomposition using the **per-pixel look vectors** (`lv_theta`/`lv_phi`), not a scalar incidence angle |

**Paper-exact windows for this winter**

| Product | Window |
|---|---|
| Descending S1A HH — SBAS | 17 Feb – 6 Apr 2019 |
| Ascending S1B VV — SBAS | 21 Feb – 22 Apr 2019 |
| 2D decomposition overlap | 21 Feb – 6 Apr 2019 (44 d) |

**Targets:** horizontal −95 → +85 cm, vertical −41 → +63 cm; coherence over ice > 0.4 only
from mid-February.

In [ ]:
# run once per machine
!pip install -q asf_search hyp3_sdk rasterio numpy scipy matplotlib pandas

## S0 — Configuration

In [ ]:
# ==========================================================================
#  S0 - CONFIGURATION
# ==========================================================================
import os, glob, json, zipfile, warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap
from matplotlib.dates import DateFormatter

import rasterio, rasterio.windows, rasterio.transform
from rasterio.warp import reproject, Resampling, transform as rio_transform
from scipy.ndimage import uniform_filter

warnings.filterwarnings('ignore')

# ---------------- this notebook's winter ---------------------------------
WINTER = 2019

CFG = {
    2019: dict(work='./insar_cambridge',
               desc_win=('2019-02-17', '2019-04-06'),   # paper: S1A descending HH
               asc_win =('2019-02-21', '2019-04-22'),   # paper: S1B ascending  VV
               overlap =('2019-02-21', '2019-04-06'),   # paper: multi-dim SBAS, 44 d
               season  =('2018-12-01', '2019-05-31')),
    2021: dict(work='./insar_cambridge_2021',
               desc_win=('2021-02-18', '2021-04-07'),
               asc_win =('2021-02-22', '2021-04-23'),
               overlap =('2021-02-22', '2021-04-07'),
               season  =('2020-12-01', '2021-05-31')),
}[WINTER]

WORK_DIR = CFG['work']
OUT_DIR  = f'./figures_{WINTER}'
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- physics and conventions --------------------------------
WAVELENGTH = 0.05546576              # m, Sentinel-1 C-band
FRINGE_CM  = WAVELENGTH / 2 * 100    # 2.77 cm range change per 2-pi colour cycle
COH_THRESH = 0.4                     # paper Section 3: coherence threshold 0.4
GRID_RES   = 80.0                    # m, HyP3 20x4 looks
REF_WIN    = 5                       # px, median window used to sample a point

# LOS sign. HyP3 unwrapped phase grows as the target moves AWAY from the sensor,
# so displacement toward the sensor = -unw * lambda / (4 pi).
# The paper's convention is "+ = movement toward sensor" -> LOS_SIGN = -1.
LOS_SIGN = -1.0

# ---------------- study area ---------------------------------------------
TOWN_LAT, TOWN_LON = 69.117, -105.052    # Cambridge Bay town, for distance reporting

# Centre of the ANALYSIS GRID. NOT the town: the ice points sit well south of
# Cambridge Bay, and a box centred on the town crops most of the channel away.
CB_LAT, CB_LON = 68.98, -105.18
AOI_HALF_KM    = (38, 30)                # -> 76 x 60 km

RP_LATLON       = (69.1169, -105.0385)   # reference point, assumed stable (paper RP)
P5_LATLON       = (68.9582, -104.9118)   # land control point (paper P5)
PROFILER_LATLON = (69.1128, -105.0596)   # Ocean Networks Canada ice profiler, CB dock

# Fixed monitoring points, identical in both winters so the cross-winter
# comparison measures the same places.
#   P1  69d 0'15.45"N  105d10'25.61"W
#   P2  68d52'12.50"N  105d 1'20.13"W
#   P3  68d54'42.09"N  105d39'22.41"W
#   P4  68d50' 2.74"N  105d13'43.19"W
# Set an entry to None to auto-place that point by ice surface class instead.
POINT_OVERRIDE = {
    'P1': (69.0043, -105.1738),
    'P2': (68.8701, -105.0223),
    'P3': (68.9117, -105.6562),
    'P4': (68.8341, -105.2287),
}

# Chen et al. (2021), Table 3 - for the side-by-side comparison in S12
PAPER = {
    'LOS descending (cm)': {'P1': -12.5, 'P2':   5.0, 'P3':  16.7, 'P4':  -5.4},
    'LOS ascending (cm)' : {'P1':  18.1, 'P2':  15.7, 'P3': -13.6, 'P4':  51.2},
    'Horizontal E-W (cm)': {'P1': -28.9, 'P2': -14.8, 'P3':  21.7, 'P4': -61.8},
    'Vertical U-D (cm)'  : {'P1':  -0.7, 'P2':   6.6, 'P3':   9.4, 'P4':   9.7},
}
PAPER_RANGE = dict(horizontal=(-95, 85), vertical=(-41, 63))

CLR = {'P1': '#0072B2', 'P2': '#E69F00', 'P3': '#009E73',
       'P4': '#D55E00', 'P5': '#7B3294', 'RP': '#333333'}
MKR = {'P1': 'o', 'P2': 's', 'P3': '^', 'P4': 'D', 'P5': 'v', 'RP': '*'}

mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'figure.facecolor': 'white',
    'axes.prop_cycle': mpl.cycler(color=list(CLR.values())),
})

print(f'winter {WINTER}   products in {WORK_DIR}   figures -> {OUT_DIR}')
print(f'analysis grid centred on {CB_LAT}N {CB_LON}W, '
      f'{2*AOI_HALF_KM[0]} x {2*AOI_HALF_KM[1]} km')
print(f'descending SBAS {CFG["desc_win"][0]} .. {CFG["desc_win"][1]}')
print(f'ascending  SBAS {CFG["asc_win"][0]} .. {CFG["asc_win"][1]}')
print(f'2D overlap      {CFG["overlap"][0]} .. {CFG["overlap"][1]}')

## S1 — Products, analysis grid, raster I/O

In [ ]:
# ==========================================================================
#  S1 - PRODUCT DISCOVERY, COMMON ANALYSIS GRID, RASTER I/O
# ==========================================================================
def _d(s): return datetime.strptime(s, '%Y-%m-%d')

DESC_WIN = tuple(_d(x) for x in CFG['desc_win'])
ASC_WIN  = tuple(_d(x) for x in CFG['asc_win'])
OVERLAP  = tuple(_d(x) for x in CFG['overlap'])
SEASON   = tuple(_d(x) for x in CFG['season'])


def find_products(base_dir):
    # one dict per HyP3 product folder, with a path for every band we may need
    prods = []
    for d in sorted(glob.glob(os.path.join(base_dir, 'S1*'))):
        if not os.path.isdir(d):
            continue
        files = os.listdir(d)
        def pick(sfx):
            m = sorted(f for f in files if f.endswith(sfx))
            return os.path.join(d, m[0]) if m else None
        unw, cor = pick('_unw_phase.tif'), pick('_corr.tif')
        if not unw or not cor:
            continue
        parts = os.path.basename(d).split('_')
        try:
            d1 = datetime.strptime(parts[1][:8], '%Y%m%d')
            d2 = datetime.strptime(parts[2][:8], '%Y%m%d')
        except Exception:
            continue
        prods.append(dict(name=os.path.basename(d), dir=d,
                          date1=min(d1, d2), date2=max(d1, d2), dt=abs((d2 - d1).days),
                          pol=parts[3][:2] if len(parts) > 3 else '??',
                          unw=unw, cor=cor,
                          wrp=pick('_wrapped_phase.tif'), amp=pick('_amp.tif'),
                          dem=pick('_dem.tif'), lvt=pick('_lv_theta.tif'),
                          lvp=pick('_lv_phi.tif'), wat=pick('_water_mask.tif'),
                          inc=pick('_inc_map.tif'), meta=pick('.txt')))
    return prods


def dedup(prods):
    # HyP3 downloads land twice - keep one product per (date1, date2)
    seen = {}
    for p in sorted(prods, key=lambda q: q['name']):
        seen.setdefault((p['date1'], p['date2']), p)
    return sorted(seen.values(), key=lambda q: (q['date1'], q['date2']))


DESC = dedup(find_products(f'{WORK_DIR}/desc'))
ASC  = dedup(find_products(f'{WORK_DIR}/asc'))
assert DESC and ASC, f'no products under {WORK_DIR} - check WORK_DIR / WINTER'
print(f'{len(DESC)} descending + {len(ASC)} ascending unique pairs')

# ---------------- one common analysis grid for BOTH orbits ----------------
with rasterio.open(DESC[0]['unw']) as _s:
    GRID_CRS = _s.crs
_xs, _ys = rio_transform('EPSG:4326', GRID_CRS, [CB_LON], [CB_LAT])
_cx, _cy = _xs[0], _ys[0]
_x0 = np.floor((_cx - AOI_HALF_KM[0]*1000) / GRID_RES) * GRID_RES
_x1 = np.ceil ((_cx + AOI_HALF_KM[0]*1000) / GRID_RES) * GRID_RES
_y0 = np.floor((_cy - AOI_HALF_KM[1]*1000) / GRID_RES) * GRID_RES
_y1 = np.ceil ((_cy + AOI_HALF_KM[1]*1000) / GRID_RES) * GRID_RES
GRID_W, GRID_H = int((_x1 - _x0) / GRID_RES), int((_y1 - _y0) / GRID_RES)
GRID_TF = rasterio.transform.from_origin(_x0, _y1, GRID_RES, GRID_RES)
EXT = [0.0, GRID_W*GRID_RES/1000.0, 0.0, GRID_H*GRID_RES/1000.0]
print(f'analysis grid {GRID_H} x {GRID_W} px @ {GRID_RES:.0f} m '
      f'= {EXT[1]:.0f} x {EXT[3]:.0f} km, CRS {GRID_CRS}')

_CACHE = {}
def read_aoi(fp, nodata=0.0):
    # read one band of a HyP3 GeoTIFF onto the common grid (bounds-aware reproject,
    # NOT scipy zoom - this is what makes ascending and descending line up)
    if fp is None:
        return None
    if fp in _CACHE:
        return _CACHE[fp].copy()
    with rasterio.open(fp) as src:
        w = rasterio.windows.from_bounds(_x0 - 2000, _y0 - 2000, _x1 + 2000, _y1 + 2000,
                                         transform=src.transform)
        w = w.round_offsets().round_lengths()
        a = src.read(1, window=w, boundless=True,
                     fill_value=(nodata if nodata is not None else 0)).astype('float32')
        stf, scrs = src.window_transform(w), src.crs
    out = np.full((GRID_H, GRID_W), np.nan, 'float32')
    reproject(a, out, src_transform=stf, src_crs=scrs,
              dst_transform=GRID_TF, dst_crs=GRID_CRS,
              src_nodata=nodata, dst_nodata=np.nan,
              resampling=Resampling.bilinear)
    _CACHE[fp] = out
    return out.copy()


def ll2rc(lat, lon):
    xs, ys = rio_transform('EPSG:4326', GRID_CRS, [lon], [lat])
    c, r = ~GRID_TF * (xs[0], ys[0])
    return int(round(r)), int(round(c))

def rc2ll(r, c):
    x, y = GRID_TF * (c + 0.5, r + 0.5)
    lon, lat = rio_transform(GRID_CRS, 'EPSG:4326', [x], [y])
    return lat[0], lon[0]

def rc2km(r, c):
    return (c + 0.5)*GRID_RES/1000.0, EXT[3] - (r + 0.5)*GRID_RES/1000.0

def sample(a, rc, win=REF_WIN):
    # median over a small window - one pixel is far too noisy over sea ice
    if a is None:
        return np.nan
    r, c = rc; h = win // 2
    b = a[max(0, r-h):r+h+1, max(0, c-h):c+h+1]
    b = b[np.isfinite(b)]
    return float(np.median(b)) if b.size else np.nan


def los_map(p, ref_rc, coh_thresh=COH_THRESH):
    # reference-corrected LOS displacement in metres. S3b REPLACES this with a
    # deramping version - that is the one S9 and S10 actually use.
    unw = read_aoi(p['unw']); coh = read_aoi(p['cor'])
    unw[unw == 0] = np.nan
    ref = sample(unw, ref_rc, 7)
    if not np.isfinite(ref):
        return None, coh
    d = LOS_SIGN * (unw - ref) * WAVELENGTH / (4 * np.pi)
    d[~np.isfinite(coh) | (coh < coh_thresh)] = np.nan
    return d, coh

print('helpers ready')

## S1b — Self-test on synthetic data

Checks the SBAS inversion, the 2D decomposition and the time interpolation against known
answers **before any real data is touched**. All three errors must be ~1e-15.

In [ ]:
# ==========================================================================
#  S1b - SELF-TEST ON SYNTHETIC DATA (no InSAR products needed)
# ==========================================================================
_ok = True

# --- SBAS: does the increment inversion recover a known signal? -----------
_dts = [datetime(2019, 2, 17) + timedelta(days=12*i) for i in range(5)]
_true_inc = np.array([0.05, -0.02, 0.11, -0.07])           # m per interval
_true_cum = np.concatenate([[0], np.cumsum(_true_inc)])
_pairs = [(i, j) for i in range(len(_dts)) for j in range(i+1, len(_dts))
          if (_dts[j] - _dts[i]).days in (12, 24, 36)]
_A = np.zeros((len(_pairs), len(_dts) - 1))
for _k, (_i, _j) in enumerate(_pairs):
    _A[_k, _i:_j] = 1.0
_x = np.linalg.pinv(_A) @ (_A @ _true_inc)
_e = float(np.abs(np.concatenate([[0], np.cumsum(_x)]) - _true_cum).max())
print(f'SBAS      : {len(_pairs)} pairs, rank {np.linalg.matrix_rank(_A)}/{len(_dts)-1}, '
      f'max error {_e:.1e} m')
_ok &= _e < 1e-9

# --- 2D decomposition round-trip -----------------------------------------
_thd, _phd = np.pi/2 - np.radians(33.5), np.radians(9.0)     # descending: sat to the EAST
_tha, _pha = np.pi/2 - np.radians(39.0), np.radians(171.0)   # ascending : sat to the WEST
_ad, _bd = np.cos(_thd)*np.cos(_phd), np.sin(_thd)
_aa, _ba = np.cos(_tha)*np.cos(_pha), np.sin(_tha)
_det = _ad*_ba - _bd*_aa
_worst = 0.0
for _dE, _dU in ((0.30, -0.10), (-0.62, 0.097), (-0.95, 0.63), (0.0, 0.0)):
    _ld, _la = _dE*_ad + _dU*_bd, _dE*_aa + _dU*_ba
    _worst = max(_worst,
                 abs(( _ld*_ba - _la*_bd)/_det - _dE),
                 abs((-_ld*_aa + _la*_ad)/_det - _dU))
print(f'2D decomp : det = {_det:.3f} (well conditioned), max error {_worst:.1e} m')
_ok &= (_worst < 1e-9 and abs(_det) > 0.1)

# --- time interpolation ---------------------------------------------------
def _it(ds, cum, tg):
    t  = np.array([(d - ds[0]).days for d in ds], float)
    tt = np.array([(d - ds[0]).days for d in tg], float)
    f = cum.reshape(len(ds), -1)
    j = np.clip(np.searchsorted(t, tt), 1, len(t) - 1)
    w = (tt - t[j-1]) / np.maximum(t[j] - t[j-1], 1e-9)
    return (f[j-1]*(1 - w[:, None]) + f[j]*w[:, None]).reshape(len(tg), *cum.shape[1:])

_src = [datetime(2019, 2, 21) + timedelta(days=12*i) for i in range(5)]
_cum = np.arange(5, dtype='float32')[:, None, None] * np.ones((5, 3, 3), 'float32')
_tg = [datetime(2019, 2, 21), datetime(2019, 3, 1), datetime(2019, 4, 6)]
_got = _it(_src, _cum, _tg)[:, 0, 0]
_exp = np.array([(d - _src[0]).days / 12 for d in _tg])
_ei = float(np.abs(_got - _exp).max())
print(f'interp    : max error {_ei:.1e}')
_ok &= _ei < 1e-6

print('\nSELF-TEST', 'PASSED' if _ok else 'FAILED - do not trust anything below')
assert _ok, 'self-test failed'

## S2 — Extra HyP3 jobs (optional)

Adds 24-day and 36-day pairs so the SBAS network has redundancy. Consecutive 12-day pairs
alone form a single chain — one bad unwrap corrupts everything after it, and 2D map coverage
stays sparse. Set `SUBMIT_EXTRA = True`, wait, then `DOWNLOAD_EXTRA = True`, then re-run S1.

In [ ]:
# ==========================================================================
#  S2 - EXTRA HyP3 JOBS FOR A REDUNDANT SBAS NETWORK  (optional)
# ==========================================================================
SUBMIT_EXTRA   = False     # <<< True once per winter to submit
DOWNLOAD_EXTRA = False     # <<< True once the jobs have SUCCEEDED
EXTRA_DT       = (24, 36)  # extra temporal baselines, days

if SUBMIT_EXTRA:
    import asf_search as asf
    from hyp3_sdk import HyP3
    from collections import Counter

    AOI = 'POLYGON((-106.0 68.8, -104.0 68.8, -104.0 69.5, -106.0 69.5, -106.0 68.8))'
    lo = (min(DESC_WIN[0], ASC_WIN[0]) - timedelta(days=40)).strftime('%Y-%m-%d')
    hi = (max(DESC_WIN[1], ASC_WIN[1]) + timedelta(days=10)).strftime('%Y-%m-%d')
    hyp3 = HyP3(prompt=True)

    for flight, sub, have in [(asf.FLIGHT_DIRECTION.DESCENDING, 'desc', DESC),
                              (asf.FLIGHT_DIRECTION.ASCENDING,  'asc',  ASC)]:
        res = asf.search(platform=[asf.PLATFORM.SENTINEL1],
                         processingLevel=asf.PRODUCT_TYPE.SLC,
                         beamMode=[asf.BEAMMODE.IW], flightDirection=flight,
                         intersectsWith=AOI, start=lo, end=hi)
        sc = sorted(res, key=lambda s: s.properties['startTime'])
        track = Counter(s.properties['pathNumber'] for s in sc).most_common(1)[0][0]
        sc = [s for s in sc if s.properties['pathNumber'] == track]
        uniq = {}
        for s in sc:
            uniq.setdefault(s.properties['startTime'][:10], s)
        scenes = [uniq[k] for k in sorted(uniq)]
        existing = {(p['date1'], p['date2']) for p in have}

        n = 0
        for i in range(len(scenes)):
            for j in range(i + 1, len(scenes)):
                a = _d(scenes[i].properties['startTime'][:10])
                b = _d(scenes[j].properties['startTime'][:10])
                if (b - a).days not in EXTRA_DT or (a, b) in existing:
                    continue
                hyp3.submit_insar_job(
                    granule1=scenes[i].properties['sceneName'],
                    granule2=scenes[j].properties['sceneName'],
                    name=f'cb{WINTER}_{sub}_{a:%m%d}_{b:%m%d}', looks='20x4',
                    include_wrapped_phase=True, include_inc_map=True,
                    include_displacement_maps=True, include_look_vectors=True,
                    apply_water_mask=False)
                n += 1
        print(f'{sub}: track {track}, {len(scenes)} scenes, submitted {n} extra jobs')
    print('\nnow wait for the jobs, then set DOWNLOAD_EXTRA = True')

if DOWNLOAD_EXTRA:
    from hyp3_sdk import HyP3
    hyp3 = HyP3(prompt=True)
    jobs = [j for j in hyp3.find_jobs() if j.name and j.name.startswith(f'cb{WINTER}_')]
    print(f'{len(jobs)} extra jobs for winter {WINTER}')
    for j in jobs:
        j = hyp3.get_job_by_id(j.job_id)
        if j.status_code != 'SUCCEEDED':
            print(f'  {j.name}: {j.status_code}')
            continue
        sub = 'desc' if '_desc_' in j.name else 'asc'
        j.download_files(f'{WORK_DIR}/{sub}')
    for sub in ('desc', 'asc'):
        for zf in glob.glob(f'{WORK_DIR}/{sub}/*.zip'):
            with zipfile.ZipFile(zf) as z:
                z.extractall(f'{WORK_DIR}/{sub}')
    print('downloaded and unzipped - RE-RUN S1 to pick up the new products')

if not SUBMIT_EXTRA and not DOWNLOAD_EXTRA:
    print('S2 skipped (both flags False) - this is normal, carry on to S3')

## S3 — Water mask, ice surface classes, monitoring points

In [ ]:
# ==========================================================================
#  S3 - WATER MASK, ICE-SURFACE CLASSES, MONITORING POINTS
#       P1/P3 smooth ice | P2 rough ice | P4 smooth-rough transition
#       (Chen et al. 2021, Section 4.2)
# ==========================================================================
from scipy.ndimage import (binary_opening, binary_closing, binary_dilation,
                           label, distance_transform_edt)


def mean_coh(prods, lo, hi, max_dt=12):
    st = [read_aoi(p['cor']) for p in prods
          if p['date1'] >= lo and p['date2'] <= hi and p['dt'] <= max_dt]
    if not st:
        st = [read_aoi(p['cor']) for p in prods if p['date1'] >= lo and p['date2'] <= hi]
    return np.nanmean(np.dstack(st), axis=2) if st else np.full((GRID_H, GRID_W), np.nan)


def first_file(prods, key):
    # HyP3 only ships a band if the job asked for it, and not every product in a
    # folder is guaranteed to have it - so search them all, not just prods[0]
    p = next((q for q in prods if q.get(key)), None)
    return p[key] if p else None


COH_WIN    = mean_coh(DESC, *DESC_WIN)                       # paper SBAS window
COH_SEASON = mean_coh(DESC, *SEASON)
COH_DECJAN = mean_coh(DESC, SEASON[0], _d(f'{SEASON[0].year + 1}-01-31'))


def build_water_mask():
    dem_fp = first_file(DESC + ASC, 'dem')
    wat_fp = first_file(DESC + ASC, 'wat')
    dem = read_aoi(dem_fp, nodata=None) if dem_fp else None
    if dem is None:
        print('no *_dem.tif (jobs lacked include_dem=True) - using coherence for land/sea')

    if wat_fp is not None:
        m = np.nan_to_num(read_aoi(wat_fp, nodata=None)) > 0.5
        if m.any() and not m.all():
            if dem is not None:
                h1, h0 = np.nanmedian(dem[m]), np.nanmedian(dem[~m])
                water = m if h1 < h0 else ~m          # sea sits at 0 m
                print(f'water mask from {os.path.basename(wat_fp)}, polarity from DEM '
                      f'(water {min(h1,h0):.1f} m, land {max(h1,h0):.1f} m)')
                return water & np.isfinite(dem)
            ref = COH_DECJAN if np.isfinite(COH_DECJAN).any() else COH_SEASON
            c1, c0 = np.nanmean(ref[m]), np.nanmean(ref[~m])
            water = m if c1 < c0 else ~m       # land stays coherent, new ice does not
            print(f'water mask from {os.path.basename(wat_fp)}, polarity from '
                  f'early-winter coherence (water {min(c1,c0):.2f}, land {max(c1,c0):.2f})')
            return water
        print('water mask is uniform over the study area - ignoring it')

    if dem is not None:
        print('no usable *_water_mask.tif - using DEM < 2 m as the sea mask')
        return np.isfinite(dem) & (dem < 2.0)
    ref = COH_DECJAN if np.isfinite(COH_DECJAN).any() else COH_SEASON
    print('no water mask and no DEM - sea mask from early-winter coherence < 0.35')
    return np.isfinite(ref) & (ref < 0.35)


def clean_water(mask, min_blob_km2=2.0):
    # the raw mask speckles land with false water pixels; those are the most coherent
    # pixels in the scene, so any "best coherence" rule lands straight on them
    px = np.ones((3, 3), bool)
    m = binary_closing(binary_opening(mask, px, iterations=2), px, iterations=2)
    lab, n = label(m)
    if n:
        min_px = int(min_blob_km2 * 1e6 / GRID_RES**2)
        sizes = np.bincount(lab.ravel()); sizes[0] = 0
        m = (sizes >= min_px)[lab]
    return m


RAW_WATER = build_water_mask()
WATER = clean_water(RAW_WATER)
print(f'water mask cleaned: {int(RAW_WATER.sum())} -> {int(WATER.sum())} px '
      f'({100*WATER.mean():.0f}% of frame)')
if not (20 < 100*WATER.mean() < 95):
    print('   WARNING: sea fraction looks wrong for Cambridge Bay (expect ~50-80%)')

# distance from the nearest shore - the depth proxy
LAND_DIST_KM = distance_transform_edt(WATER) * GRID_RES / 1000.0
print(f'open water reaches {LAND_DIST_KM.max():.1f} km from the nearest shore')

# distance and azimuth from the TOWN, for reporting only
_r0, _c0 = ll2rc(TOWN_LAT, TOWN_LON)
_rr, _cc = np.mgrid[0:GRID_H, 0:GRID_W]
DIST_KM = np.hypot(_rr - _r0, _cc - _c0) * GRID_RES / 1000.0
AZ_DEG  = (np.degrees(np.arctan2(_cc - _c0, _r0 - _rr)) + 360) % 360


def snap_to_land(lat, lon, rad=15):
    r0, c0 = ll2rc(lat, lon)
    best, bv = (r0, c0), -np.inf
    for r in range(max(0, r0-rad), min(GRID_H, r0+rad+1)):
        for c in range(max(0, c0-rad), min(GRID_W, c0+rad+1)):
            if WATER[r, c]:
                continue
            v = COH_SEASON[r, c]
            if np.isfinite(v) and v > bv:
                bv, best = v, (r, c)
    return best


def describe(n, tag, rc):
    r, c = rc; lat, lon = rc2ll(r, c)
    return dict(point=n, surface=tag, lat=round(lat, 4), lon=round(lon, 4),
                km_from_land=round(float(LAND_DIST_KM[r, c]), 1),
                km_from_town=round(float(DIST_KM[r, c]), 1),
                az_deg=int(AZ_DEG[r, c]),
                coh_FebApr=round(float(COH_WIN[r, c]), 2),
                coh_DecJan=round(float(COH_DECJAN[r, c]), 2),
                on_water=bool(WATER[r, c]))


# --------------------------------------------- ice surface classes (Sec 4.2)
# smooth ice is dark and homogeneous on the SAR image, rough ice bright and
# patchy; the transition between them is where the paper sees the largest strain
# keep points away from the raster boundary: the 5x5 sampling window runs off the
# edge there and the deramp/filter misbehave
MARGIN_PX = int(round(3000 / GRID_RES))
IN_FRAME = np.zeros((GRID_H, GRID_W), bool)
IN_FRAME[MARGIN_PX:-MARGIN_PX, MARGIN_PX:-MARGIN_PX] = True

ICE = (WATER & IN_FRAME & np.isfinite(COH_WIN) & (COH_WIN > COH_THRESH)
       & (~np.isfinite(COH_DECJAN) | (COH_DECJAN < COH_THRESH))
       & (LAND_DIST_KM >= 1.5) & (DIST_KM < 40))

_ap = next((p for p in DESC if p['date1'] >= DESC_WIN[0] and p['amp']), None)
if _ap is not None and ICE.any():
    _amp = read_aoi(_ap['amp'])
    _db  = 10 * np.log10(np.where(_amp > 0, _amp, np.nan))
    _mn  = uniform_filter(np.nan_to_num(_db), 5)
    _tx  = np.sqrt(np.maximum(uniform_filter(np.nan_to_num(_db)**2, 5) - _mn**2, 0))
    _lo, _hi = np.nanpercentile(_mn[ICE], [35, 65])
    _tmed    = np.nanpercentile(_tx[ICE], 50)
    SMOOTH = ICE & (_mn <= _lo) & (_tx <= _tmed)
    ROUGH  = ICE & (_mn >= _hi) & (_tx >  _tmed)
    TRANS  = ICE & binary_dilation(SMOOTH, iterations=4) & binary_dilation(ROUGH, iterations=4)
    print(f'ice classes from {os.path.basename(_ap["amp"])}: '
          f'smooth {int(SMOOTH.sum())} px, rough {int(ROUGH.sum())} px, '
          f'transition {int(TRANS.sum())} px')
else:
    print('no amplitude image - falling back to a plain shore-distance transect')
    SMOOTH = ROUGH = TRANS = ICE

if not TRANS.any():
    print('   no smooth/rough transition found - P4 falls back to the rough class')
    TRANS = ROUGH


def place_points():
    # (name, class, where along the shore-distance range, label)
    spec = [('P1', SMOOTH, 'min', 'smooth ice, near coast'),
            ('P2', ROUGH,  'mid', 'rough ice'),
            ('P3', SMOOTH, 'max', 'smooth ice, offshore'),
            ('P4', TRANS,  'mid', 'smooth/rough transition')]
    pts, info, used = {}, [], []
    for n, cls, where, tag in spec:
        if POINT_OVERRIDE.get(n):
            pts[n] = ll2rc(*POINT_OVERRIDE[n])
            info.append(describe(n, 'manual', pts[n])); used.append(pts[n]); continue
        cand = cls.copy()
        for ur, uc in used:                       # keep the points at least 4 km apart
            cand &= np.hypot(_rr - ur, _cc - uc) * GRID_RES / 1000.0 > 4.0
        if not cand.any():
            print(f'  !! {n}: no pixel left in its ice class - '
                  f'set POINT_OVERRIDE["{n}"] by hand')
            continue
        rs, cs = np.nonzero(cand)
        dd = LAND_DIST_KM[rs, cs]
        k = int(np.argmin(dd) if where == 'min' else
                np.argmax(dd) if where == 'max' else
                np.abs(dd - np.median(dd)).argmin())
        pts[n] = (int(rs[k]), int(cs[k])); used.append(pts[n])
        info.append(describe(n, tag, pts[n]))

    for n, ll in (('RP', RP_LATLON), ('P5', P5_LATLON)):
        pts[n] = snap_to_land(*ll)
        info.append(describe(n, 'land', pts[n]))
    return pts, pd.DataFrame(info)


PTS, PT_TABLE = place_points()
PT_TABLE.to_csv(f'{OUT_DIR}/table_points_{WINTER}.csv', index=False)
print()
print(PT_TABLE.to_string(index=False))

print('\nsanity check - P1-P4 must be on water and NOT coherent in Dec/Jan:')
for n in ('P1', 'P2', 'P3', 'P4'):
    if n not in PTS:
        continue
    r, c = PTS[n]
    flags = []
    if not WATER[r, c]:                 flags.append('ON LAND')
    if LAND_DIST_KM[r, c] < 1.5:        flags.append('too close to shore')
    if COH_DECJAN[r, c] >= COH_THRESH:  flags.append('coherent in Dec/Jan')
    print(f'  {n}  {LAND_DIST_KM[r, c]:4.1f} km offshore, Dec-Jan coh '
          f'{COH_DECJAN[r, c]:.2f}   {"OK" if not flags else "<-- " + ", ".join(flags)}')

# ---- freeze helper -------------------------------------------------------
# Auto-placement depends on THIS winter's amplitude scene, so the two winters
# land on different points and the cross-winter comparison becomes meaningless.
# Paste the dict below into POINT_OVERRIDE in S0 of BOTH notebooks.
if not any(POINT_OVERRIDE.values()):
    print('\n*** POINT_OVERRIDE is empty - P1-P4 were auto-placed from this winter\'s')
    print('*** imagery, so the OTHER winter will pick DIFFERENT locations and the')
    print('*** cross-winter comparison will not be valid. Freeze them with:')
print()
print('POINT_OVERRIDE = {')
for n in ('P1', 'P2', 'P3', 'P4'):
    if n in PTS:
        _la, _lo2 = rc2ll(*PTS[n])
        print(f"    '{n}': ({_la:.4f}, {_lo2:.4f}),")
print('}')

## S3b — Deramping

Land cannot deform, so any long-wavelength phase gradient over stable coherent land is
atmosphere, ionosphere or orbit. Fit it there, subtract it everywhere. This is the stand-in
for the split-spectrum ionospheric correction the paper used and HyP3 does not do.

**This cell redefines `los_map`** — S9 and S10 resolve it at call time, so nothing else changes.
The median residual over land is your measurement uncertainty; quote it.

In [ ]:
# ==========================================================================
#  S3b - REMOVE LONG-WAVELENGTH RAMPS
# ==========================================================================
from scipy.ndimage import binary_erosion

DERAMP       = True
DERAMP_ORDER = 1        # 1 = plane (recommended), 2 = quadratic
MIN_LAND_PX  = 500      # minimum stable-land pixels needed to trust a fit

LAND_STABLE = binary_erosion((~WATER) & np.isfinite(COH_SEASON) & (COH_SEASON > 0.6),
                             iterations=2)
print(f'deramp reference surface: {int(LAND_STABLE.sum())} stable land pixels '
      f'({100*LAND_STABLE.mean():.0f}% of frame)')

_Rn = (np.repeat(np.arange(GRID_H, dtype='float32')[:, None], GRID_W, 1) - GRID_H/2) / GRID_H
_Cn = (np.repeat(np.arange(GRID_W, dtype='float32')[None, :], GRID_H, 0) - GRID_W/2) / GRID_W
_cols = [np.ones_like(_Rn), _Rn, _Cn]
if DERAMP_ORDER >= 2:
    _cols += [_Rn*_Rn, _Cn*_Cn, _Rn*_Cn]
_BASIS = np.dstack(_cols)
DERAMP_LOG = []


def los_map(p, ref_rc, coh_thresh=COH_THRESH):
    # 1) deramp against stable land  2) zero at RP (paper convention)  3) to metres
    unw = read_aoi(p['unw']); coh = read_aoi(p['cor'])
    unw[unw == 0] = np.nan

    if DERAMP:
        m = LAND_STABLE & np.isfinite(unw) & np.isfinite(coh) & (coh > 0.5)
        if m.sum() >= MIN_LAND_PX:
            coef, *_ = np.linalg.lstsq(_BASIS[m], unw[m], rcond=None)
            ramp = _BASIS @ coef
            k = WAVELENGTH / (4 * np.pi) * 100          # radians -> cm
            DERAMP_LOG.append(dict(
                pair=f'{p["date1"]:%d%b}-{p["date2"]:%d%b}', n_land=int(m.sum()),
                ramp_pp_cm=round(float(ramp.max() - ramp.min()) * k, 1),
                land_rms_cm=round(float(np.sqrt(np.mean((unw[m] - ramp[m])**2))) * k, 2)))
            unw = unw - ramp
        else:
            DERAMP_LOG.append(dict(pair=f'{p["date1"]:%d%b}-{p["date2"]:%d%b}',
                                   n_land=int(m.sum()), ramp_pp_cm=np.nan,
                                   land_rms_cm=np.nan))

    ref = sample(unw, ref_rc, 7)
    if not np.isfinite(ref):
        return None, coh
    d = LOS_SIGN * (unw - ref) * WAVELENGTH / (4 * np.pi)
    d[~np.isfinite(coh) | (coh < coh_thresh)] = np.nan
    return d, coh


DERAMP_LOG.clear()
for _p in [q for q in DESC if q['date1'] >= DESC_WIN[0] and q['date2'] <= DESC_WIN[1]]:
    los_map(_p, PTS['RP'])
_dl = pd.DataFrame(DERAMP_LOG).drop_duplicates('pair')
_dl.to_csv(f'{OUT_DIR}/table_deramp_{WINTER}.csv', index=False)
print()
print(_dl.to_string(index=False))
print(f'\nmedian ramp removed        {_dl.ramp_pp_cm.median():.1f} cm peak-to-peak')
print(f'median residual over land  {_dl.land_rms_cm.median():.2f} cm  '
      f'<- quote this as your measurement uncertainty')
DERAMP_LOG.clear()

## S4 — Interferogram inventory

In [ ]:
# ==========================================================================
#  S4 - INVENTORY OF EVERY INTERFEROGRAM
# ==========================================================================
def bperp(p):
    if not p.get('meta') or not os.path.exists(p['meta']):
        return np.nan
    try:
        for line in open(p['meta'], errors='ignore'):
            if 'baseline' in line.lower() and ':' in line:
                for tok in line.split(':')[-1].split():
                    try:
                        return float(tok)
                    except ValueError:
                        continue
    except Exception:
        pass
    return np.nan


def inventory(prods, label, win):
    rows = []
    for p in prods:
        coh = read_aoi(p['cor'])
        ice = coh[WATER & np.isfinite(coh)]
        rows.append(dict(
            orbit=label, pol=p['pol'],
            date1=p['date1'].date(), date2=p['date2'].date(), dt_days=p['dt'],
            bperp_m=round(bperp(p), 1),
            mean_ice_coh=round(float(np.nanmean(ice)), 3) if ice.size else np.nan,
            frac_ice_coh_gt_0p4=round(float((ice > COH_THRESH).mean()), 3) if ice.size else np.nan,
            in_sbas_window=bool(p['date1'] >= win[0] and p['date2'] <= win[1])))
    return pd.DataFrame(rows)


INV = pd.concat([inventory(DESC, 'descending', DESC_WIN),
                 inventory(ASC,  'ascending',  ASC_WIN)], ignore_index=True)
INV.to_csv(f'{OUT_DIR}/table_inventory_{WINTER}.csv', index=False)
print(INV.to_string(index=False))
_nd = int(((INV.orbit == 'descending') & INV.in_sbas_window).sum())
_na = int(((INV.orbit == 'ascending')  & INV.in_sbas_window).sum())
print(f'\ninside the SBAS windows: {_nd} descending, {_na} ascending')
if _nd < 3 or _na < 3:
    print('WARNING: very few pairs in the window - check WINTER / WORK_DIR')

## S5 — Figure 1: study area

In [ ]:
# ==========================================================================
#  S5 - PLOT HELPERS AND FIGURE 1 (STUDY AREA)
# ==========================================================================
XK = (np.arange(GRID_W) + 0.5) * GRID_RES / 1000.0
YK = EXT[3] - (np.arange(GRID_H) + 0.5) * GRID_RES / 1000.0


def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(f'{OUT_DIR}/{name}_{WINTER}.{ext}', dpi=300,
                    bbox_inches='tight', facecolor='white')
    print(f'  saved {OUT_DIR}/{name}_{WINTER}.png')


def add_land(ax, color='#c4c4c4', zorder=2):
    ax.imshow(np.where(WATER, np.nan, 1.0), extent=EXT, origin='upper',
              cmap=ListedColormap([color]), vmin=0, vmax=1,
              zorder=zorder, interpolation='nearest')


def add_coast(ax, color='#ffd21f', lw=1.0, zorder=4):
    ax.contour(XK, YK[::-1], WATER[::-1].astype(float), levels=[0.5],
               colors=color, linewidths=lw, zorder=zorder)


def add_points(ax, names=('P1', 'P2', 'P3', 'P4'), color='k', label=True, ms=7):
    for n in names:
        if n not in PTS:
            continue
        x, y = rc2km(*PTS[n])
        ax.plot(x, y, MKR[n], ms=ms, mfc='none', mew=1.8, color=color, zorder=6)
        if label:
            ax.annotate(n, (x, y), xytext=(6, 5), textcoords='offset points',
                        fontsize=9, fontweight='bold', color=color, zorder=7,
                        path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])


def add_scalebar(ax, km=10):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x = x0 + 0.06*(x1 - x0); y = y0 + 0.07*(y1 - y0)
    ax.plot([x, x + km], [y, y], lw=3.5, color='k', solid_capstyle='butt', zorder=8,
            path_effects=[pe.withStroke(linewidth=6, foreground='white')])
    ax.text(x + km/2, y + 0.02*(y1 - y0), f'{km} km', ha='center', va='bottom',
            fontsize=8, fontweight='bold', zorder=8,
            path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])


def add_north(ax):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    x = x1 - 0.08*(x1 - x0); y = y0 + 0.08*(y1 - y0)
    ax.annotate('N', xy=(x, y + 0.10*(y1 - y0)), xytext=(x, y), ha='center', va='bottom',
                fontsize=11, fontweight='bold', zorder=8,
                arrowprops=dict(arrowstyle='-|>', color='k', lw=1.6))


def finish_map(ax, title=None, xlabel=True, ylabel=True):
    for s in ax.spines.values():
        s.set_visible(True)
    ax.set_xlim(EXT[0], EXT[1]); ax.set_ylim(EXT[2], EXT[3]); ax.set_aspect('equal')
    if title:
        ax.set_title(title)
    ax.set_xlabel('Easting (km)' if xlabel else '')
    ax.set_ylabel('Northing (km)' if ylabel else '')
    if not xlabel: ax.set_xticklabels([])
    if not ylabel: ax.set_yticklabels([])


# ---------------------------------------------------------------- Figure 1
_ref = next((p for p in DESC if p['date1'] >= DESC_WIN[0] and p['amp']), DESC[0])
amp = read_aoi(_ref['amp'])
db = 10 * np.log10(np.where(amp > 0, amp, np.nan))

fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(db, extent=EXT, origin='upper', cmap='gray',
          vmin=np.nanpercentile(db, 3), vmax=np.nanpercentile(db, 97), zorder=1)
add_coast(ax)
add_points(ax, ('P1', 'P2', 'P3', 'P4'), color='#ff2d2d')
for n, col, mk, lab in [('RP', '#00e5ff', '*', 'RP  reference'),
                        ('P5', '#c77dff', 'v', 'P5  land')]:
    if n in PTS:
        x, y = rc2km(*PTS[n])
        ax.plot(x, y, mk, ms=13 if n == 'RP' else 8, color=col, mew=1.6,
                mfc='none' if n != 'RP' else col, zorder=6, label=lab)
        ax.annotate(n, (x, y), xytext=(7, 5), textcoords='offset points', color=col,
                    fontsize=9, fontweight='bold', zorder=7,
                    path_effects=[pe.withStroke(linewidth=2.5, foreground='black')])
px, py = rc2km(*ll2rc(*PROFILER_LATLON))
ax.plot(px, py, 'X', ms=9, color='#39ff14', mew=1.2, zorder=6, label='ice profiler (SWIP)')
ax.plot([], [], 'r'+MKR['P1'], mfc='none', label='P1-P4  sea-ice points')
add_scalebar(ax); add_north(ax)
finish_map(ax, f'Cambridge Bay study area — Sentinel-1 amplitude '
               f'{_ref["date1"]:%d %b %Y}\n(cf. Chen et al. 2021, Figure 1)')
ax.legend(loc='upper left', fontsize=8, framealpha=0.75, frameon=True)
plt.tight_layout(); save(fig, 'fig01_study_area'); plt.show()

## S6 — Coherence over time (paper §4.1)

In [ ]:
# ==========================================================================
#  S6 - COHERENCE: TIME SERIES, AREA FRACTION, MAPS, INTERVAL TABLE
#       the ONE section that keeps December, exactly as the paper does
# ==========================================================================
rows = []
for lab, prods in (('descending', DESC), ('ascending', ASC)):
    for p in prods:
        coh = read_aoi(p['cor'])
        ice = coh[WATER & np.isfinite(coh)]
        rec = dict(orbit=lab, mid=p['date1'] + timedelta(days=p['dt'] // 2), dt=p['dt'])
        for n in ('P1', 'P2', 'P3', 'P4'):
            rec[n] = sample(coh, PTS[n]) if n in PTS else np.nan
        rec['area_frac'] = float((ice > COH_THRESH).mean()) if ice.size else np.nan
        rows.append(rec)
COH_TS = pd.DataFrame(rows).sort_values('mid')
COH_TS.to_csv(f'{OUT_DIR}/table_coherence_timeseries_{WINTER}.csv', index=False)

d12 = COH_TS[(COH_TS.orbit == 'descending') & (COH_TS.dt == 12)]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True,
                             gridspec_kw=dict(height_ratios=[2, 1], hspace=0.12))
for n in ('P1', 'P2', 'P3', 'P4'):
    a1.plot(d12['mid'], d12[n], marker=MKR[n], color=CLR[n], lw=1.8, ms=7, label=n)
a1.axhline(COH_THRESH, color='crimson', ls='--', lw=1.6)
a1.text(0.012, COH_THRESH + 0.02, 'coherence threshold 0.4', color='crimson',
        fontsize=9, transform=a1.get_yaxis_transform())
a1.axvspan(DESC_WIN[0], DESC_WIN[1], color='#2ca02c', alpha=0.10, lw=0)
a1.text(DESC_WIN[0] + (DESC_WIN[1] - DESC_WIN[0]) / 2, 1.03, 'SBAS window',
        ha='center', fontsize=9, color='#2ca02c', fontweight='bold')
a1.set_ylim(0, 1.12); a1.set_ylabel('Coherence at P1–P4')
a1.set_title(f'Sentinel-1 descending 12-day coherence over landfast ice, winter {WINTER}\n'
             f'(cf. Chen et al. 2021, §4.1 — C-band only coherent from mid-February)')
a1.legend(ncol=4, loc='upper left')

a2.plot(d12['mid'], 100 * d12['area_frac'], 'o-', color='#333333', lw=1.8, ms=6)
a2.axvspan(DESC_WIN[0], DESC_WIN[1], color='#2ca02c', alpha=0.10, lw=0)
a2.set_ylabel('Ice area with\ncoherence > 0.4 (%)'); a2.set_xlabel('Date')
a2.set_ylim(0, 100); a2.grid(alpha=0.25)
a2.xaxis.set_major_formatter(DateFormatter('%d %b'))
plt.tight_layout(); save(fig, 'fig02_coherence_timeseries'); plt.show()

# ---- coherence maps: early / mid / late winter ---------------------------
def nearest(prods, when, dt=12):
    cand = [p for p in prods if p['dt'] == dt] or prods
    return min(cand, key=lambda p: abs((p['date1'] - when).days))

yr = SEASON[0].year
picks = [nearest(DESC, datetime(yr, 12, 15)),
         nearest(DESC, datetime(yr + 1, 1, 20)),
         nearest(DESC, DESC_WIN[0]),
         nearest(DESC, DESC_WIN[1] - timedelta(days=12))]
seen, panels = set(), []
for p in picks:
    if p['name'] not in seen:
        seen.add(p['name']); panels.append(p)

fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.6))
for k, (ax, p) in enumerate(zip(np.atleast_1d(axes), panels)):
    coh = read_aoi(p['cor'])
    im = ax.imshow(coh, extent=EXT, origin='upper', cmap='magma', vmin=0, vmax=1, zorder=1)
    add_coast(ax, '#66ccff', 0.7)
    add_points(ax, color='w', label=(k == 0))
    finish_map(ax, f'{p["date1"]:%d %b} – {p["date2"]:%d %b} ({p["dt"]} d)',
               ylabel=(k == 0))
    if k == len(panels) - 1:
        plt.colorbar(im, ax=np.atleast_1d(axes), label='Coherence', shrink=0.85, pad=0.02)
fig.suptitle(f'Descending coherence through winter {WINTER} — '
             f'high along the coast, decreasing toward the open channel', y=1.02)
save(fig, 'fig03_coherence_maps'); plt.show()

# ---- Table 2 equivalent: coherence vs temporal baseline ------------------
late = COH_TS[(COH_TS.orbit == 'descending') & (COH_TS['mid'] >= DESC_WIN[0])]
if late.dt.nunique() > 1:
    T2 = late.groupby('dt')[['P1', 'P2', 'P3', 'P4']].mean().round(2)
    T2['n_pairs'] = late.groupby('dt').size()
    T2.index.name = 'interval (days)'
    T2.to_csv(f'{OUT_DIR}/table2_coherence_vs_interval_{WINTER}.csv')
    print('\nCoherence vs temporal baseline, mid-Feb onward (cf. paper Table 2):')
    print(T2.to_string())
else:
    print('\nOnly one temporal baseline present — run S2 (SUBMIT_EXTRA) to build the '
          'coherence-vs-interval table.')

## S7 — Wrapped interferograms (paper Fig. 5 and 6)

In [ ]:
# ==========================================================================
#  S7 - WRAPPED INTERFEROGRAMS, PAPER-EXACT WINDOWS
#       one colour cycle (-pi to pi) = 2.8 cm range change
# ==========================================================================
def wrapped(p):
    if p['wrp'] and os.path.exists(p['wrp']):
        w = read_aoi(p['wrp'])
    else:
        u = read_aoi(p['unw']); u[u == 0] = np.nan
        w = np.angle(np.exp(1j * u))
    coh = read_aoi(p['cor'])
    w[~np.isfinite(coh) | (coh < COH_THRESH)] = np.nan
    return w


def ifg_panel(prods, win, title, fname, ncol=2):
    sel = [p for p in prods if p['date1'] >= win[0] and p['date2'] <= win[1]]
    sel = sorted(sel, key=lambda p: (p['date1'], p['dt']))
    seen, use = set(), []
    for p in sel:                       # one panel per start date, shortest baseline
        if p['date1'] not in seen:
            seen.add(p['date1']); use.append(p)
    use = use[:4]
    if not use:
        print(f'no interferograms inside {win[0]:%d %b} - {win[1]:%d %b}'); return
    nrow = int(np.ceil(len(use) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.4 * ncol, 5.2 * nrow), squeeze=False)
    cyc = plt.get_cmap('hsv').copy(); cyc.set_bad(alpha=0.0)
    for k, p in enumerate(use):
        ax = axes.flat[k]
        add_land(ax)
        im = ax.imshow(wrapped(p), extent=EXT, origin='upper', cmap=cyc,
                       vmin=-np.pi, vmax=np.pi, zorder=3)
        add_coast(ax, '#222222', 0.6)
        add_points(ax, color='k', label=(k == 0))
        finish_map(ax, f'({chr(65+k)})  {p["date1"]:%d %b} – {p["date2"]:%d %b %Y}'
                       f'   ({p["dt"]} d)',
                   xlabel=(k // ncol == nrow - 1), ylabel=(k % ncol == 0))
        if k == 0:
            add_scalebar(ax); add_north(ax)
    for k in range(len(use), nrow * ncol):
        axes.flat[k].axis('off')
    cb = fig.colorbar(im, ax=axes, shrink=0.6, pad=0.02, ticks=[-np.pi, 0, np.pi])
    cb.ax.set_yticklabels(['−π', '0', 'π'])
    cb.set_label(f'Wrapped phase — one cycle = {FRINGE_CM:.1f} cm range change')
    fig.suptitle(title, y=1.0)
    save(fig, fname); plt.show()


ifg_panel(DESC, DESC_WIN,
          f'Sentinel-1 descending HH wrapped interferograms, winter {WINTER}\n'
          f'(cf. Chen et al. 2021, Figure 5) — grey = land, white = coherence < 0.4',
          'fig04_ifg_descending')

ifg_panel(ASC, ASC_WIN,
          f'Sentinel-1 ascending VV wrapped interferograms, winter {WINTER}\n'
          f'(cf. Chen et al. 2021, Figure 6) — grey = land, white = coherence < 0.4',
          'fig05_ifg_ascending')

# early-winter contrast: dense parallel fringes before February (paper §4.2)
early = [p for p in DESC if p['date2'] <= _d(f'{SEASON[0].year + 1}-02-01')]
late  = [q for q in DESC if q['date1'] >= DESC_WIN[0] and q['date2'] <= DESC_WIN[1]]
if not early:
    print('no pre-February descending pairs - skipping the fringe-evolution figure')
elif not late:
    print('no descending pairs inside the SBAS window - skipping the fringe-evolution figure')
else:
    p_e, p_l = early[len(early)//2], late[-1]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
    cyc = plt.get_cmap('hsv').copy(); cyc.set_bad(alpha=0.0)
    for ax, p, tag in ((axes[0], p_e, 'early winter — dense parallel fringes,\nstrong lateral deformation'),
                       (axes[1], p_l, 'late winter — sparse, partly circular fringes,\nstable thick ice')):
        add_land(ax)
        im = ax.imshow(wrapped(p), extent=EXT, origin='upper', cmap=cyc,
                       vmin=-np.pi, vmax=np.pi, zorder=3)
        add_coast(ax, '#222222', 0.6); add_points(ax, color='k')
        finish_map(ax, f'{p["date1"]:%d %b} – {p["date2"]:%d %b %Y}\n{tag}')
    fig.suptitle('Fringe pattern evolution over the winter (cf. Chen et al. 2021, §4.2)', y=1.02)
    save(fig, 'fig06_fringe_evolution'); plt.show()

## S8 — Goldstein phase filter

In [ ]:
# ==========================================================================
#  S8 - GOLDSTEIN ADAPTIVE PHASE FILTER (noise reduction demonstration)
# ==========================================================================
def goldstein_filter(phase, patch=32, alpha=0.5, overlap=8):
    ph = np.nan_to_num(phase)
    cpx = np.exp(1j * ph); out = np.zeros_like(cpx); wt = np.zeros(ph.shape)
    step = patch - overlap
    for i in range(0, ph.shape[0] - patch + 1, step):
        for j in range(0, ph.shape[1] - patch + 1, step):
            spec = np.fft.fft2(cpx[i:i+patch, j:j+patch])
            w = uniform_filter(np.abs(spec), 3) ** alpha
            out[i:i+patch, j:j+patch] += np.fft.ifft2(spec * w)
            wt[i:i+patch, j:j+patch] += 1
    wt[wt == 0] = 1
    res = np.angle(out / wt)
    res[~np.isfinite(phase)] = np.nan
    return res


pg = [p for p in DESC if p['date1'] >= DESC_WIN[0] and p['date2'] <= DESC_WIN[1]][-1]
raw = wrapped(pg)
filt = goldstein_filter(raw)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
cyc = plt.get_cmap('hsv').copy(); cyc.set_bad(alpha=0.0)
for ax, arr, tag in ((axes[0], raw, 'before — HyP3 wrapped phase'),
                     (axes[1], filt, 'after — Goldstein filter (α = 0.5, 32 px)')):
    add_land(ax)
    im = ax.imshow(arr, extent=EXT, origin='upper', cmap=cyc, vmin=-np.pi, vmax=np.pi, zorder=3)
    add_coast(ax, '#222222', 0.6); add_points(ax, color='k')
    finish_map(ax, tag)
cb = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02, ticks=[-np.pi, 0, np.pi])
cb.ax.set_yticklabels(['−π', '0', 'π']); cb.set_label(f'one cycle = {FRINGE_CM:.1f} cm')
fig.suptitle(f'Goldstein phase filtering — {pg["date1"]:%d %b} – {pg["date2"]:%d %b %Y}', y=1.02)
save(fig, 'fig07_goldstein'); plt.show()

## S9 — SBAS at the monitoring points (paper Fig. 8 and 9)

In [ ]:
# ==========================================================================
#  S9 - SBAS TIME SERIES AT P1-P5
#       unknowns = displacement increments between consecutive dates
#       A x = d_los, solved by pinv (SVD -> minimum-norm least squares)
# ==========================================================================
def sbas_points(prods, win, label):
    P = [p for p in prods if p['date1'] >= win[0] and p['date2'] <= win[1]]
    dates = sorted({d for p in P for d in (p['date1'], p['date2'])})
    idx = {d: i for i, d in enumerate(dates)}; nd = len(dates)
    if nd < 2:
        print(f'{label}: fewer than 2 dates in the window'); return {}, dates

    A_rows, vals, wts, used = [], {n: [] for n in PTS}, {n: [] for n in PTS}, []
    for p in P:
        d, coh = los_map(p, PTS['RP'])
        if d is None:
            print(f'  skipped {p["name"]}: reference point incoherent'); continue
        row = np.zeros(nd - 1); row[idx[p['date1']]:idx[p['date2']]] = 1.0
        A_rows.append(row); used.append(p)
        for n, rc in PTS.items():
            vals[n].append(sample(d, rc)); wts[n].append(sample(coh, rc))
    A_all = np.array(A_rows)
    print(f'{label}: {len(used)} interferograms, {nd} dates '
          f'({dates[0]:%d %b} .. {dates[-1]:%d %b}), '
          f'network rank {np.linalg.matrix_rank(A_all)}/{nd-1}')

    out = {}
    for n in PTS:
        v = np.array(vals[n], float); w = np.nan_to_num(np.array(wts[n], float))
        good = np.isfinite(v) & (w >= COH_THRESH)
        if good.sum() < 1 or np.linalg.matrix_rank(A_all[good]) < nd - 1:
            out[n] = dict(dates=dates, cum=np.full(nd, np.nan), n=int(good.sum()), rms=np.nan)
            continue
        A, b, ww = A_all[good], v[good], w[good]
        x = np.linalg.pinv(A * ww[:, None]) @ (b * ww)
        out[n] = dict(dates=dates, cum=np.concatenate([[0.0], np.cumsum(x)]),
                      n=int(good.sum()),
                      rms=float(np.sqrt(np.mean((A @ x - b) ** 2))) * 100)
    return out, dates


SBAS_D, DATES_D = sbas_points(DESC, DESC_WIN, 'descending')
SBAS_A, DATES_A = sbas_points(ASC,  ASC_WIN,  'ascending')

# ---- Figures 8 and 9 ----------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), sharey=True)
for ax, sb, tag, pol in ((axes[0], SBAS_D, 'Descending (S1A)', 'HH'),
                         (axes[1], SBAS_A, 'Ascending (S1B)',  'VV')):
    for n in ('P1', 'P2', 'P3', 'P4', 'P5'):
        if n not in sb or not np.isfinite(sb[n]['cum']).any():
            continue
        ax.plot(sb[n]['dates'], sb[n]['cum'] * 100, marker=MKR[n], color=CLR[n],
                lw=2, ms=7, label=n if n != 'P5' else 'P5 (land)')
    ax.axhline(0, color='k', lw=0.8)
    ax.grid(alpha=0.25)
    ax.set_xlabel('Date'); ax.set_title(f'{tag} — {pol} polarisation')
    ax.xaxis.set_major_formatter(DateFormatter('%d %b'))
    for lb in ax.get_xticklabels():
        lb.set_rotation(30); lb.set_ha('right')
axes[0].set_ylabel('Cumulative LOS displacement (cm)\n(+ toward sensor, − away)')
axes[0].legend(ncol=3)
fig.suptitle(f'Sentinel-1 SBAS cumulative LOS deformation, winter {WINTER}\n'
             f'(cf. Chen et al. 2021, Figures 8 and 9) — all values relative to RP', y=1.02)
plt.tight_layout(); save(fig, 'fig08_los_timeseries'); plt.show()

# ---- verification: the land point must be flat --------------------------
print('\nVERIFICATION - P5 is on land and must stay near zero:')
for tag, sb in (('descending', SBAS_D), ('ascending', SBAS_A)):
    if 'P5' in sb and np.isfinite(sb['P5']['cum']).any():
        mx = np.nanmax(np.abs(sb['P5']['cum'])) * 100
        print(f'  {tag}: max |P5| = {mx:5.1f} cm   '
              f'{"OK" if mx < 5 else "<-- check RP location / LOS_SIGN"}')

## S10 — Raster SBAS and 2D decomposition

In [ ]:
# ==========================================================================
#  S10 - PER-PIXEL SBAS OVER THE WHOLE FRAME, THEN 2D DECOMPOSITION
# ==========================================================================
def sbas_raster(prods, win, label):
    P = [p for p in prods if p['date1'] >= win[0] and p['date2'] <= win[1]]
    dates = sorted({d for p in P for d in (p['date1'], p['date2'])})
    idx = {d: i for i, d in enumerate(dates)}; nd = len(dates)
    if nd < 2:
        raise RuntimeError(f'{label}: only {nd} acquisition date(s) inside '
                           f'{win[0]:%d %b %Y} - {win[1]:%d %b %Y}')
    A, D = [], []
    for p in P:
        d, _ = los_map(p, PTS['RP'])
        if d is None:
            continue
        row = np.zeros(nd - 1); row[idx[p['date1']]:idx[p['date2']]] = 1.0
        A.append(row); D.append(d)
    if not D:
        raise RuntimeError(f'{label}: no interferogram in the window is coherent at RP')
    A = np.asarray(A); D = np.dstack(D)
    npair, npix = A.shape[0], GRID_H * GRID_W
    Dm = D.reshape(npix, npair).T
    valid = np.isfinite(Dm)

    X = np.full((nd - 1, npix), np.nan, 'float32')
    rng = np.random.RandomState(0).rand(npair)
    key = (valid * rng[:, None]).sum(axis=0)
    uk, inv = np.unique(key, return_inverse=True)
    solved = 0
    for k in range(uk.size):
        sel = inv == k
        m = valid[:, np.argmax(sel)]
        if m.sum() < 1 or np.linalg.matrix_rank(A[m]) < nd - 1:
            continue
        X[:, sel] = (np.linalg.pinv(A[m]) @ Dm[np.ix_(m, sel)]).astype('float32')
        solved += sel.sum()
    cum = np.concatenate([np.zeros((1, npix), 'float32'), np.cumsum(X, axis=0)])
    print(f'{label}: {npair} pairs, {nd} dates, '
          f'{100*solved/npix:.0f}% of pixels have a full-rank sub-network')
    return dates, cum.reshape(nd, GRID_H, GRID_W)


def interp_time(dates_src, cum, targets):
    if len(dates_src) < 2:
        raise RuntimeError('interp_time needs at least 2 source dates')
    t  = np.array([(d - dates_src[0]).days for d in dates_src], float)
    tt = np.array([(d - dates_src[0]).days for d in targets], float)
    flat = cum.reshape(len(dates_src), -1)
    j = np.clip(np.searchsorted(t, tt), 1, len(t) - 1)
    w = (tt - t[j-1]) / np.maximum(t[j] - t[j-1], 1e-9)
    out = flat[j-1] * (1 - w[:, None]) + flat[j] * w[:, None]
    return out.reshape(len(targets), *cum.shape[1:])


DATES_RD, CUM_D = sbas_raster(DESC, DESC_WIN, 'descending raster SBAS')
DATES_RA, CUM_A = sbas_raster(ASC,  ASC_WIN,  'ascending  raster SBAS')

# ---- per-pixel look vectors (HyP3 lv_theta / lv_phi) --------------------
# lv_theta is the look-vector ELEVATION angle (90 deg - incidence). Sentinel-1
# incidence (30-45 deg) and elevation (45-60 deg) overlap around 0.79 rad, so we
# settle it against inc_map.tif, which is unambiguously the incidence angle.
def look_vectors(prods, label):
    p = next((q for q in prods if q['lvt'] and q['lvp']), None)
    if p is None:
        raise RuntimeError(f'{label}: no lv_theta/lv_phi - resubmit with '
                           'include_look_vectors=True')
    th = read_aoi(p['lvt'], nodata=None)
    ph = read_aoi(p['lvp'], nodata=None)
    if np.nanmax(np.abs(th)) > 3.2:
        th, ph = np.radians(th), np.radians(ph)

    inc_p = next((q for q in prods if q['inc']), None)
    if inc_p is not None:
        inc = read_aoi(inc_p['inc'], nodata=None)
        if np.nanmax(np.abs(inc)) > 3.2:
            inc = np.radians(inc)
        m = np.isfinite(th) & np.isfinite(inc)
        as_elev = np.nanmedian(np.abs(th[m] - (np.pi/2 - inc[m])))
        as_inc  = np.nanmedian(np.abs(th[m] - inc[m]))
        if as_inc < as_elev:
            print(f'{label}: lv_theta matches inc_map -> it IS incidence, converting')
            th = np.pi/2 - th
        else:
            print(f'{label}: lv_theta confirmed as elevation angle '
                  f'(residual vs 90-inc = {np.degrees(as_elev):.2f} deg)')
    else:
        print(f'{label}: no inc_map to cross-check lv_theta - assuming elevation angle')

    inc_deg = np.degrees(np.pi/2 - np.nanmedian(th))
    print(f'{label}: incidence {inc_deg:.1f} deg, ground->satellite azimuth '
          f'{np.degrees(np.nanmedian(ph)):.1f} deg CCW from east')
    if not (20 < inc_deg < 55):
        print(f'   WARNING: incidence {inc_deg:.1f} deg is outside the Sentinel-1 IW range')
    return th, ph


TH_D, PH_D = look_vectors(DESC, 'descending')
TH_A, PH_A = look_vectors(ASC,  'ascending')

_cd, _ca = float(np.nanmedian(np.cos(PH_D))), float(np.nanmedian(np.cos(PH_A)))
print(f'east-west sensitivity: descending {_cd:+.2f}, ascending {_ca:+.2f}')
if _cd * _ca > 0:
    print('   *** WARNING: both orbits look the same way - the decomposition is '
          'ill-posed and its horizontal/vertical values are NOT meaningful ***')

# ground -> satellite unit vector u = [cos(th)cos(ph), cos(th)sin(ph), sin(th)]
# with no north-south motion (near-polar orbits are blind to it, paper Section 3):
#     d_los = dE * cos(th)cos(ph) + dU * sin(th)
AD, BD = np.cos(TH_D) * np.cos(PH_D), np.sin(TH_D)
AA, BA = np.cos(TH_A) * np.cos(PH_A), np.sin(TH_A)
DET = AD * BA - BD * AA
DET = np.where(np.abs(DET) < 1e-3, np.nan, DET)


def decompose(los_d, los_a):
    dE = ( los_d * BA - los_a * BD) / DET
    dU = (-los_d * AA + los_a * AD) / DET
    return dE, dU


# ---- common time axis over the paper's overlap window -------------------
TARGETS = [d for d in DATES_RD if OVERLAP[0] <= d <= OVERLAP[1]][:6]
if len(TARGETS) < 2:
    raise RuntimeError(f'only {len(TARGETS)} descending date(s) in the overlap window')

# the reference epoch is interpolated but NOT drawn - it would be an all-zero panel
LOS_D_T = interp_time(DATES_RD, CUM_D, [OVERLAP[0]] + TARGETS)
LOS_D_T = LOS_D_T[1:] - LOS_D_T[0]
LOS_A_T = interp_time(DATES_RA, CUM_A, [OVERLAP[0]] + TARGETS)
LOS_A_T = LOS_A_T[1:] - LOS_A_T[0]

DE = np.empty_like(LOS_D_T); DU = np.empty_like(LOS_D_T)
for k in range(len(TARGETS)):
    DE[k], DU[k] = decompose(LOS_D_T[k], LOS_A_T[k])
ICE_M = WATER[None, :, :]
DE = np.where(ICE_M, DE, np.nan); DU = np.where(ICE_M, DU, np.nan)

print(f'\npanels: {" | ".join(f"{d:%d %b}" for d in TARGETS)}')
print(f'horizontal E-W : {np.nanpercentile(DE[-1]*100, 1):7.1f} to '
      f'{np.nanpercentile(DE[-1]*100, 99):6.1f} cm   '
      f'(paper {PAPER_RANGE["horizontal"][0]} to {PAPER_RANGE["horizontal"][1]} cm)')
print(f'vertical  U-D  : {np.nanpercentile(DU[-1]*100, 1):7.1f} to '
      f'{np.nanpercentile(DU[-1]*100, 99):6.1f} cm   '
      f'(paper {PAPER_RANGE["vertical"][0]} to {PAPER_RANGE["vertical"][1]} cm)')

## S11 — Horizontal and vertical deformation maps (paper Fig. 10 and 11)

In [ ]:
# ==========================================================================
#  S11 - FIGURES 10 AND 11: CUMULATIVE HORIZONTAL AND VERTICAL DEFORMATION
# ==========================================================================
def deform_panels(stack, targets, title, cbar_label, fname, cap=None):
    n = len(targets)
    ncol = 3 if n > 2 else n
    nrow = int(np.ceil(n / ncol))
    if cap is not None:
        v = float(cap)
    elif np.isfinite(stack[-1]).any():
        v = float(max(np.nanpercentile(np.abs(stack[-1] * 100), 98), 1.0))
    else:
        v = 50.0
        print(f'   WARNING: {fname} has no valid pixels in the final panel - '
              f'using a fixed +/-{v:.0f} cm scale')
    cmap = plt.get_cmap('RdBu_r').copy(); cmap.set_bad(alpha=0.0)

    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.6 * nrow), squeeze=False)
    for k in range(nrow * ncol):
        ax = axes.flat[k]
        if k >= n:
            ax.axis('off'); continue
        add_land(ax)
        im = ax.imshow(stack[k] * 100, extent=EXT, origin='upper', cmap=cmap,
                       vmin=-v, vmax=v, zorder=3)
        add_coast(ax, '#333333', 0.6)
        add_points(ax, color='k', label=(k == 0))
        finish_map(ax, f'({chr(65+k)})  {OVERLAP[0]:%d %b} – {targets[k]:%d %b %Y}',
                   xlabel=(k // ncol == nrow - 1), ylabel=(k % ncol == 0))
        if k == 0:
            add_scalebar(ax); add_north(ax)
    cb = fig.colorbar(im, ax=axes, shrink=0.65, pad=0.02)
    cb.set_label(cbar_label)
    fig.suptitle(title, y=1.0)
    save(fig, fname); plt.show()


span = (f'{OVERLAP[0]:%d %b} – {TARGETS[-1]:%d %b %Y} '
        f'({(TARGETS[-1]-OVERLAP[0]).days} days)')

deform_panels(DE, TARGETS,
              f'Cumulative horizontal (west–east) deformation of landfast ice, {span}\n'
              f'(cf. Chen et al. 2021, Figure 10) — grey = land, white = no measurement',
              'Horizontal displacement (cm)   + eastward / − westward',
              'fig09_horizontal_maps')

deform_panels(DU, TARGETS,
              f'Cumulative vertical (up–down) deformation of landfast ice, {span}\n'
              f'(cf. Chen et al. 2021, Figure 11) — grey = land, white = no measurement',
              'Vertical displacement (cm)   + upward / − downward',
              'fig10_vertical_maps')

## S12 — Time series, Table 3, export

In [ ]:
# ==========================================================================
#  S12 - FIGURES 12/13, TABLE 3 COMPARISON, EXPORT
# ==========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharex=True)
for ax, stack, tag, ylab in (
        (axes[0], DE, 'Horizontal (west–east)', 'Cumulative horizontal (cm)\n(+ east, − west)'),
        (axes[1], DU, 'Vertical (up–down)',     'Cumulative vertical (cm)\n(+ up, − down)')):
    for n in ('P1', 'P2', 'P3', 'P4', 'P5'):
        if n not in PTS:
            continue
        r, c = PTS[n]
        series = np.array([sample(stack[k], (r, c)) for k in range(len(TARGETS))]) * 100
        if not np.isfinite(series).any():
            continue
        ax.plot(TARGETS, series, marker=MKR[n], color=CLR[n], lw=2, ms=7,
                label=n if n != 'P5' else 'P5 (land)')
    ax.axhline(0, color='k', lw=0.8); ax.grid(alpha=0.25)
    ax.set_title(tag); ax.set_ylabel(ylab); ax.set_xlabel('Date')
    ax.xaxis.set_major_formatter(DateFormatter('%d %b'))
    for lb in ax.get_xticklabels():
        lb.set_rotation(30); lb.set_ha('right')
axes[0].legend(ncol=3)
fig.suptitle(f'Sentinel-1 cumulative horizontal and vertical deformation at P1–P5, {span}\n'
             f'(cf. Chen et al. 2021, Figures 12 and 13)', y=1.03)
plt.tight_layout(); save(fig, 'fig11_hv_timeseries'); plt.show()

# ---- Table 3 equivalent, with a raster fallback -------------------------
res = {}
for n in ('P1', 'P2', 'P3', 'P4'):
    if n not in PTS:
        continue
    r, c = PTS[n]
    d_los = SBAS_D[n]['cum'][-1]*100 if n in SBAS_D else np.nan
    a_los = SBAS_A[n]['cum'][-1]*100 if n in SBAS_A else np.nan
    fell_back = []
    if not np.isfinite(d_los):
        d_los = float(sample(LOS_D_T[-1], (r, c))) * 100; fell_back.append('desc')
    if not np.isfinite(a_los):
        a_los = float(sample(LOS_A_T[-1], (r, c))) * 100; fell_back.append('asc')
    if fell_back:
        print(f'  {n}: point-SBAS network incomplete for {"/".join(fell_back)} '
              f'- value taken from the raster solution (overlap window)')
    res[n] = {
        'LOS descending (cm)': d_los,
        'LOS ascending (cm)' : a_los,
        'Horizontal E-W (cm)': sample(DE[-1], (r, c)) * 100,
        'Vertical U-D (cm)'  : sample(DU[-1], (r, c)) * 100,
    }

rows = []
for meas in PAPER:
    rows.append({'measurement': meas, 'source': f'this study {WINTER}',
                 **{n: round(res[n][meas], 1) for n in res}})
    rows.append({'measurement': meas, 'source': 'Chen et al. 2021',
                 **{n: PAPER[meas].get(n, np.nan) for n in res}})
T3 = pd.DataFrame(rows)
T3.to_csv(f'{OUT_DIR}/table3_deformation_{WINTER}.csv', index=False)
print('Cumulative InSAR measurements (cf. Chen et al. 2021, Table 3)')
print(f'LOS: + toward sensor | horizontal: + eastward | vertical: + upward\n')
print(T3.to_string(index=False))

summary = dict(
    winter=WINTER,
    desc_window=[str(d.date()) for d in DESC_WIN],
    asc_window=[str(d.date()) for d in ASC_WIN],
    overlap=[str(d.date()) for d in OVERLAP],
    panel_dates=[str(d.date()) for d in TARGETS],
    deramp=bool(DERAMP), deramp_order=int(DERAMP_ORDER),
    n_pairs_desc=int(sum(1 for p in DESC if DESC_WIN[0] <= p['date1'] and p['date2'] <= DESC_WIN[1])),
    n_pairs_asc=int(sum(1 for p in ASC if ASC_WIN[0] <= p['date1'] and p['date2'] <= ASC_WIN[1])),
    horizontal_range_cm=[float(np.nanpercentile(DE[-1]*100, 1)),
                         float(np.nanpercentile(DE[-1]*100, 99))],
    vertical_range_cm=[float(np.nanpercentile(DU[-1]*100, 1)),
                       float(np.nanpercentile(DU[-1]*100, 99))],
    points={n: dict(lat=round(rc2ll(*PTS[n])[0], 4), lon=round(rc2ll(*PTS[n])[1], 4),
                    **{k: (None if not np.isfinite(v) else round(float(v), 1))
                       for k, v in res[n].items()}) for n in res},
    p5_max_abs_los_cm={
        'descending': float(np.nanmax(np.abs(SBAS_D['P5']['cum'])) * 100) if 'P5' in SBAS_D else None,
        'ascending':  float(np.nanmax(np.abs(SBAS_A['P5']['cum'])) * 100) if 'P5' in SBAS_A else None},
)
with open(f'results_{WINTER}.json', 'w') as f:
    json.dump(summary, f, indent=2)

# things S13/S14 need from BOTH winters
np.savez_compressed(f'results_{WINTER}_maps.npz', de=DE[-1].astype('float32'),
                    du=DU[-1].astype('float32'), water=WATER, ext=np.array(EXT))
COH_TS.to_csv(f'results_{WINTER}_coh.csv', index=False)

print(f'\nwrote results_{WINTER}.json, results_{WINTER}_maps.npz, '
      f'results_{WINTER}_coh.csv and {OUT_DIR}/*.csv')
print(f'figures in {OUT_DIR}:')
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith('.png'):
        print('   ', f)

## S13 / S14 — Cross-winter comparison

Run **both** notebooks first (this one and
`InSAR_CambridgeBay_FINAL_2021.ipynb`), then run these two cells from either one.

In [ ]:
# ==========================================================================
#  S13 - 2018-19 vs 2020-21, numbers
# ==========================================================================
have = {}
for y in (2019, 2021):
    fp = f'results_{y}.json'
    if os.path.exists(fp):
        have[y] = json.load(open(fp))

if len(have) < 2:
    print(f'have {sorted(have)} - run the other winter\'s notebook first')
else:
    rows = []
    for y, s in have.items():
        for n, v in s['points'].items():
            rows.append(dict(winter=y, point=n,
                             **{k: v[k] for k in v if k not in ('lat', 'lon')}))
    CMP = pd.DataFrame(rows)
    CMP.to_csv('table_cross_winter.csv', index=False)
    print(CMP.to_string(index=False))

    meas = ['LOS descending (cm)', 'LOS ascending (cm)',
            'Horizontal E-W (cm)', 'Vertical U-D (cm)']
    pts = sorted(CMP.point.unique())
    x = np.arange(len(pts)); w = 0.38
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8))
    for ax, m in zip(axes.flat, meas):
        for i, (y, col) in enumerate(zip(sorted(have), ('#0072B2', '#D55E00'))):
            vals = [CMP[(CMP.winter == y) & (CMP.point == p)][m].values for p in pts]
            vals = [v[0] if len(v) else np.nan for v in vals]
            ax.bar(x + (i - 0.5)*w, vals, w, color=col, label=f'{y-1}–{y}')
        ax.axhline(0, color='k', lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels(pts); ax.set_title(m)
        ax.grid(alpha=0.2, axis='y')
    axes.flat[0].legend()
    fig.suptitle('Cambridge Bay landfast-ice deformation: 2018–19 vs 2020–21\n'
                 'Sentinel-1 SBAS, paper-equivalent windows', y=1.02)
    plt.tight_layout()
    for e in ('png', 'pdf'):
        fig.savefig(f'fig12_cross_winter.{e}', dpi=300, bbox_inches='tight',
                    facecolor='white')
    plt.show()

    print('\nranges:')
    for y, s in sorted(have.items()):
        h, v = s['horizontal_range_cm'], s['vertical_range_cm']
        print(f'  {y-1}-{y}: horizontal {h[0]:6.1f} to {h[1]:6.1f} cm | '
              f'vertical {v[0]:6.1f} to {v[1]:6.1f} cm')
    print(f'  paper  : horizontal  -95.0 to   85.0 cm | vertical  -41.0 to   63.0 cm')

In [ ]:
# ==========================================================================
#  S14 - TALK FIGURE: coherence onset + horizontal deformation, both winters
# ==========================================================================
hv = {y: np.load(f'results_{y}_maps.npz') for y in (2019, 2021)
      if os.path.exists(f'results_{y}_maps.npz')}
cohs = {y: pd.read_csv(f'results_{y}_coh.csv', parse_dates=['mid']) for y in hv
        if os.path.exists(f'results_{y}_coh.csv')}

if len(hv) < 2:
    print(f'have {sorted(hv)} - run the other winter first')
else:
    fig = plt.figure(figsize=(16, 5.2))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1, 1], wspace=0.28)

    ax0 = fig.add_subplot(gs[0])
    for y, col in zip(sorted(cohs), ('#0072B2', '#D55E00')):
        d = cohs[y]
        d = d[(d.orbit == 'descending') & (d.dt == 12)].sort_values('mid')
        doy = [(m - pd.Timestamp(f'{y-1}-12-01')).days for m in d['mid']]
        ax0.plot(doy, 100*d['area_frac'], 'o-', color=col, lw=2, ms=6, label=f'{y-1}–{y}')
    ax0.axhline(50, color='0.6', lw=0.8, ls=':')
    ax0.set_xlabel('Days since 1 December')
    ax0.set_ylabel('Ice area with coherence > 0.4 (%)')
    ax0.set_title('Coherence onset'); ax0.set_ylim(0, 100)
    ax0.grid(alpha=0.25); ax0.legend()

    vmax = max(float(np.nanpercentile(np.abs(hv[y]['de']*100), 98)) for y in hv)
    cmap = plt.get_cmap('RdBu_r').copy(); cmap.set_bad(alpha=0.0)
    maxes = []
    for k, y in enumerate(sorted(hv)):
        ax = fig.add_subplot(gs[k+1]); maxes.append(ax)
        ext = list(hv[y]['ext'])
        ax.imshow(np.where(hv[y]['water'], np.nan, 1.0), extent=ext, origin='upper',
                  cmap=ListedColormap(['#c4c4c4']), vmin=0, vmax=1, zorder=2)
        im = ax.imshow(hv[y]['de']*100, extent=ext, origin='upper', cmap=cmap,
                       vmin=-vmax, vmax=vmax, zorder=3)
        ax.set_aspect('equal'); ax.set_xlabel('Easting (km)')
        ax.set_title(f'Horizontal deformation {y-1}–{y}')
        if k == 0:
            ax.set_ylabel('Northing (km)')
        for s in ax.spines.values():
            s.set_visible(True)
    cb = fig.colorbar(im, ax=maxes, shrink=0.88, pad=0.02)
    cb.set_label('cm    + eastward / − westward')
    fig.suptitle('Cambridge Bay landfast sea ice — Sentinel-1 InSAR, two winters', y=1.03)
    for e in ('png', 'pdf'):
        fig.savefig(f'fig_talk_cross_winter.{e}', dpi=300, bbox_inches='tight',
                    facecolor='white')
    plt.show()

## Verification checklist

1. **S3b — the deramp table.** Median residual over land is your measurement uncertainty.
   If the median ramp removed is only a few mm, ramps were not a problem here.
2. **S9 — P5 (land) must stay within about ±2 cm.** If it drifts, the RP location or the
   land mask is wrong and nothing downstream is trustworthy.
3. **S3 — P1–P4 must be on water with Dec–Jan coherence < 0.4.** A point coherent in
   December is on land.
4. **S6 — coherence over ice should cross 0.4 only from mid-February** (paper §4.1).
5. **S7 — dense parallel fringes Dec–Jan, sparser and partly circular late March** (§4.2).
6. **S10/S11 — horizontal ≈ −95 → +85 cm, vertical ≈ −41 → +63 cm.** Order of magnitude is
   the realistic target.
7. **S12 — Table 3 signs and ordering should match the paper.** Its own cross-sensor
   differences were 2–44 cm, so that is the tolerance on magnitudes.

**Known limitations to state in the write-up:** no split-spectrum ionospheric correction
(deramping is the mitigation); a non-redundant SBAS network unless S2 is run; P1–P4 placed by
ice surface class rather than the paper's exact coordinates; Sentinel-1B failed in December
2021, so extending beyond winter 2020–21 loses the ascending orbit and with it the 2D
decomposition.